In [72]:
import os
import json
import random
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader

In [42]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [43]:
base_dir = "/content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M"
preprocessed_dir = os.path.join(base_dir, "AntM2C_preprocessed")
fusion_artifacts_dir = os.path.join(base_dir, "fusion_artifacts")
meta_artifacts_dir = os.path.join(base_dir, "meta_artifacts")
checkpoints_dir = os.path.join(base_dir, "checkpoints")

os.makedirs(checkpoints_dir, exist_ok=True)

print("base_dir:", base_dir)
print("preprocessed_dir:", preprocessed_dir)
print("fusion_artifacts_dir:", fusion_artifacts_dir)
print("meta_artifacts_dir:", meta_artifacts_dir)
print("checkpoints_dir:", checkpoints_dir)

base_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M
preprocessed_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed
fusion_artifacts_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/fusion_artifacts
meta_artifacts_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/meta_artifacts
checkpoints_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/checkpoints


In [44]:
# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("Reproducibility settings applied (seed =", SEED, ")")

Reproducibility settings applied (seed = 42 )


In [45]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [46]:
# Load configuration files
fusion_config_path = os.path.join(fusion_artifacts_dir, "fusion_config.json")
meta_config_path = os.path.join(meta_artifacts_dir, "meta_config.json")

assert os.path.exists(fusion_config_path), f"Missing fusion_config.json at {fusion_config_path}"
assert os.path.exists(meta_config_path), f"Missing meta_config.json at {meta_config_path}"

with open(fusion_config_path, "r") as f:
    fusion_config = json.load(f)

with open(meta_config_path, "r") as f:
    meta_config = json.load(f)

In [47]:
print("\n--- Loaded fusion_config.json ---")
for k in ["num_users", "num_items", "num_scenes", "d_model", "num_experts", "hidden_dim"]:
    print(f"{k}: {fusion_config[k]}")


--- Loaded fusion_config.json ---
num_users: 67615
num_items: 59142
num_scenes: 5
d_model: 256
num_experts: 4
hidden_dim: 256


In [48]:
print("\n--- Loaded meta_config.json ---")
for k in ["meta_dim", "FEWSHOT_N", "blend_strategy", "cold_rule"]:
    print(f"{k}: {meta_config[k]}")


--- Loaded meta_config.json ---
meta_dim: 256
FEWSHOT_N: 10
blend_strategy: deterministic_alpha
cold_rule: item_only_fewshot


### Load Core Training Artifacts

Here we will loads the precomputed, row-aligned artifacts required for end-to-end CTR training. Interaction-level features (IDs, text embeddings, and time features) must have identical length to guarantee that each row index refers to the same user–item–scene interaction across all modalities. Item-level image embeddings are loaded separately and indexed via item_id_idx.

In [49]:
# paths to artifacts
df_id_path = os.path.join(preprocessed_dir, "split_features", "df_id_encoded.pkl")
text_embeddings_path = os.path.join(preprocessed_dir, "text_embeddings.npy")
time_features_path = os.path.join(preprocessed_dir, "time_features_final.pt")

image_embeddings_path = os.path.join(base_dir, "image_embeddings_vit_base_aligned.pt")
image_missing_mask_path = os.path.join(base_dir, "image_missing_mask.pt")

splits_dir = os.path.join(preprocessed_dir, "splits")
train_idx_path = os.path.join(splits_dir, "train_idx.npy")
val_idx_path   = os.path.join(splits_dir, "val_idx.npy")
test_idx_path  = os.path.join(splits_dir, "test_idx.npy")

In [50]:
# validate file existance
required_paths = {
    "df_id_encoded.pkl": df_id_path,
    "text_embeddings.npy": text_embeddings_path,
    "time_features_final.pt": time_features_path,
    "image_embeddings_vit_base_aligned.pt": image_embeddings_path,
    "image_missing_mask.pt": image_missing_mask_path,
    "train_idx.npy": train_idx_path,
    "val_idx.npy": val_idx_path,
    "test_idx.npy": test_idx_path
}

for name, path in required_paths.items():
    assert os.path.exists(path), f"Missing {name} at: {path}"

print("All required artifacts found.")

All required artifacts found.


In [51]:
# Load interaction-level IDs (row-aligned)
df_id = pd.read_pickle(df_id_path)
print("Loaded df_id_encoded.pkl:", df_id.shape)
print("Columns:", df_id.columns.tolist())
print(df_id.head())

Loaded df_id_encoded.pkl: (3730329, 3)
Columns: ['user_id_idx', 'item_id_idx', 'scene_idx']
   user_id_idx  item_id_idx  scene_idx
0        41688        13590          4
1        39834        21351          2
2        22354        33398          2
3        17547        11190          1
4        55873        33398          2


In [52]:
# Load text embeddings as memmap (interaction-level)
text_embeddings = np.load(text_embeddings_path, mmap_mode="r")
print("Loaded text_embeddings.npy (memmap):", text_embeddings.shape, "| dtype:", text_embeddings.dtype)

Loaded text_embeddings.npy (memmap): (3730329, 1536) | dtype: float32


In [53]:
# Load time features (interaction-level)
time_features = torch.load(time_features_path, map_location="cpu")
print("Loaded time_features_final.pt:", tuple(time_features.shape), "| dtype:", time_features.dtype)

Loaded time_features_final.pt: (3730329, 5) | dtype: torch.float32


In [54]:
# Load image embeddingd + missing mask (item-level)
image_embeddings = torch.load(image_embeddings_path, map_location="cpu")
image_missing_mask = torch.load(image_missing_mask_path, map_location="cpu")

print("Loaded image_embeddings:", tuple(image_embeddings.shape), "| dtype:", image_embeddings.dtype)
print("Loaded image_missing_mask:", tuple(image_missing_mask.shape), "| missing count:", int(image_missing_mask.sum().item()))

Loaded image_embeddings: (59142, 768) | dtype: torch.float32
Loaded image_missing_mask: (59142,) | missing count: 2


In [55]:
# Load deterministic splits
train_idx = np.load(train_idx_path)
val_idx   = np.load(val_idx_path)
test_idx  = np.load(test_idx_path)

print("Loaded splits:")
print("train:", len(train_idx), "val:", len(val_idx), "test:", len(test_idx))

Loaded splits:
train: 2984263 val: 373033 test: 373033


In [56]:
N = len(df_id)

assert text_embeddings.shape[0] == N, f"text_embeddings rows ({text_embeddings.shape[0]}) != df_id rows ({N})"
assert time_features.shape[0] == N, f"time_features rows ({time_features.shape[0]}) != df_id rows ({N})"

EXPECTED_N = 3_730_329
assert N == EXPECTED_N, f"Unexpected df_id row count: {N} (expected {EXPECTED_N})"

# Split indices bounds check
assert train_idx.max() < N and val_idx.max() < N and test_idx.max() < N, "Split indices exceed dataset length"

print("All artifacts loaded and row-alignment verified.")
print(f"Interaction rows N = {N:,} (expected {EXPECTED_N:,})")

All artifacts loaded and row-alignment verified.
Interaction rows N = 3,730,329 (expected 3,730,329)


### Dataset and DataLoader

*   This section defines a PyTorch CTRDataset that returns all model inputs in a sample format.
*   Each sample is indexed by an interaction row_idx and returns the aligned ID indices (user/item/scene), interaction-level text and time features, item-level image embeddings (with missing images handled safely), and the click label.
*  A deterministic cold-start mask is also created using train-only frequency counts (few-shot rule), to allow cold/warm evaluation during training.

In [65]:
# Load labels (click) from a row-aligned source
label = None
label_source = None

# df_processed.pkl (contains label column)
df_processed_path = os.path.join(preprocessed_dir, "df_processed.pkl")
if os.path.exists(df_processed_path):
    df_processed = pd.read_pickle(df_processed_path)
    for cand in ["label", "is_click", "click", "y"]:
        if cand in df_processed.columns:
            label = df_processed[cand].to_numpy(dtype=np.float32)
            label_source = f"df_processed.pkl[{cand}]"
            break

In [68]:
assert label is not None, (
    "Could not find a label column. Expected one of ['label','is_click','click','y'] in "
    "df_processed.pkl or split_features/df_deep.csv. Please confirm where the click label lives."
)

assert len(label) == len(df_id), f"Label length {len(label)} != df_id rows {len(df_id)}"
print(f"Loaded label from: {label_source} | shape={label.shape} | positives={label.mean():.6f}")

Loaded label from: df_processed.pkl[label] | shape=(3730329,) | positives=0.336333


In [70]:
# Train-only frequency counts (for few-shot cold mask)
num_items = fusion_config["num_items"]
FEWSHOT_N = int(meta_config["FEWSHOT_N"])  # reuse what was saved in Notebook 07

train_items = df_id.iloc[train_idx]["item_id_idx"].to_numpy(dtype=np.int64)
item_train_count = np.bincount(train_items, minlength=num_items)

# Few-shot definition: seen in train but < FEWSHOT_N exposures
item_is_fewshot = (item_train_count > 0) & (item_train_count < FEWSHOT_N)
item_is_warm = (item_train_count >= FEWSHOT_N)

print("--- Train-only item frequency summary ---")
print("FEWSHOT_N:", FEWSHOT_N)
print("Few-shot items:", int(item_is_fewshot.sum()), "/", num_items)
print("Warm items:", int(item_is_warm.sum()), "/", num_items)

--- Train-only item frequency summary ---
FEWSHOT_N: 10
Few-shot items: 39751 / 59142
Warm items: 14898 / 59142


In [73]:
# convert mask to torch for fast indexing in Dataset
item_is_fewshot_torch = torch.tensor(item_is_fewshot, dtype=torch.bool)

In [74]:
class CTRDataset(Dataset):
    """
    Row-aligned CTR dataset.

    Returns:
      user_id_idx: LongTensor ()
      item_id_idx: LongTensor ()
      scene_idx:   LongTensor ()
      text_vec:    FloatTensor (1536)
      time_vec:    FloatTensor (5)
      image_vec:   FloatTensor (768) (zero if missing)
      label:       FloatTensor ()
      cold_mask:   BoolTensor ()
    """
    def __init__(
        self,
        row_indices: np.ndarray,
        df_id: pd.DataFrame,
        text_embeddings_memmap: np.ndarray,
        time_features: torch.Tensor,
        image_embeddings: torch.Tensor,
        image_missing_mask: torch.Tensor,
        labels: np.ndarray,
        item_is_fewshot: torch.Tensor
    ):
        self.row_indices = row_indices.astype(np.int64)
        self.df_id = df_id
        self.text_mem = text_embeddings_memmap
        self.time = time_features                    # CPU tensor (N,5)
        self.image = image_embeddings                # CPU tensor (num_items,768)
        self.img_missing = image_missing_mask.bool() # CPU tensor (num_items,)
        self.labels = labels                         # numpy (N,)
        self.item_is_fewshot = item_is_fewshot       # torch bool (num_items,)

        # Cache dims
        self.text_dim = self.text_mem.shape[1]
        self.time_dim = self.time.shape[1]
        self.image_dim = self.image.shape[1]

    def __len__(self):
        return len(self.row_indices)

    def __getitem__(self, i: int):
        row_idx = int(self.row_indices[i])

        row = self.df_id.iloc[row_idx]
        user_idx = int(row["user_id_idx"])
        item_idx = int(row["item_id_idx"])
        scene_idx = int(row["scene_idx"])

        # Interaction-level features
        text_vec = torch.from_numpy(self.text_mem[row_idx]).float()   # (1536,)
        time_vec = self.time[row_idx].float()                         # (5,)

        # Item-level image embedding (zero if missing)
        if bool(self.img_missing[item_idx].item()):
            image_vec = torch.zeros(self.image_dim, dtype=torch.float32)
        else:
            image_vec = self.image[item_idx].float()                  # (768,)

        # Label
        y = float(self.labels[row_idx])
        label = torch.tensor(y, dtype=torch.float32)

        # Cold mask (item-only few-shot rule)
        cold_mask = self.item_is_fewshot[item_idx]  # bool scalar

        return {
            "row_idx": torch.tensor(row_idx, dtype=torch.long),
            "user_idx": torch.tensor(user_idx, dtype=torch.long),
            "item_idx": torch.tensor(item_idx, dtype=torch.long),
            "scene_idx": torch.tensor(scene_idx, dtype=torch.long),
            "text_vec": text_vec,
            "time_vec": time_vec,
            "image_vec": image_vec,
            "label": label,
            "cold_mask": cold_mask
        }


def make_loader(row_indices: np.ndarray, batch_size: int, shuffle: bool):
    ds = CTRDataset(
        row_indices=row_indices,
        df_id=df_id,
        text_embeddings_memmap=text_embeddings,
        time_features=time_features,
        image_embeddings=image_embeddings,
        image_missing_mask=image_missing_mask,
        labels=label,
        item_is_fewshot=item_is_fewshot_torch
    )

    loader = DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=4,        # adjust 2–8
        pin_memory=True,
        drop_last=False
    )
    return ds, loader

* Text embeddings are read via memmap, it avoids loading 3.7M × 1536 into RAM.
* Image embeddings are item-level and small enough to keep on CPU and index quickly. -> Can move image_vec to GPU per batch during training.
* num_workers=4 set as default on Colab; can increase to 8 only if it’s stable.

In [76]:
# Choose an initial batch size depending on device
batch_size = 4096 if device.type == "cuda" else 1024

train_ds, train_loader = make_loader(train_idx, batch_size=batch_size, shuffle=True)
val_ds, val_loader = make_loader(val_idx, batch_size=batch_size, shuffle=False)

print("Datasets created:")
print("Train size:", len(train_ds), "| Val size:", len(val_ds))
print("Batch size:", batch_size)

Datasets created:
Train size: 2984263 | Val size: 373033
Batch size: 1024


In [77]:
def assert_finite(t: torch.Tensor, name: str):
    assert torch.isfinite(t).all(), f"{name} has NaN/Inf"

# First batch sanity check
batch = next(iter(train_loader))

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/tmp/ipython-input-4176152185.py:52: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:206.)
  text_vec = torch.from_numpy(self.text_mem[row_idx]).float()   # (1536,)
/tmp/ipython-input-4176152185.py:52: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to pro

In [78]:
# Shapes
print("user_idx:", batch["user_idx"].shape, batch["user_idx"].dtype)
print("item_idx:", batch["item_idx"].shape, batch["item_idx"].dtype)
print("scene_idx:", batch["scene_idx"].shape, batch["scene_idx"].dtype)

print("text_vec:", batch["text_vec"].shape, batch["text_vec"].dtype)
print("time_vec:", batch["time_vec"].shape, batch["time_vec"].dtype)
print("image_vec:", batch["image_vec"].shape, batch["image_vec"].dtype)

print("label:", batch["label"].shape, batch["label"].dtype)
print("cold_mask:", batch["cold_mask"].shape, batch["cold_mask"].dtype)

user_idx: torch.Size([1024]) torch.int64
item_idx: torch.Size([1024]) torch.int64
scene_idx: torch.Size([1024]) torch.int64
text_vec: torch.Size([1024, 1536]) torch.float32
time_vec: torch.Size([1024, 5]) torch.float32
image_vec: torch.Size([1024, 768]) torch.float32
label: torch.Size([1024]) torch.float32
cold_mask: torch.Size([1024]) torch.bool


In [79]:
# NaN checks (float tensors)
assert_finite(batch["text_vec"], "text_vec")
assert_finite(batch["time_vec"], "time_vec")
assert_finite(batch["image_vec"], "image_vec")
assert_finite(batch["label"], "label")

In [81]:
# Quick rates
cold_rate = float(batch["cold_mask"].float().mean().cpu())
pos_rate = float(batch["label"].mean().cpu())
print(f"Batch cold rate: {cold_rate:.4%}")
print(f"Batch positive rate: {pos_rate:.4%}")

Batch cold rate: 3.6133%
Batch positive rate: 34.6680%


In [85]:
!git commit -m "training:  dataset pipeline was validated successfully"